# NEURALSHIELD AI // PHASE 1: EXPLORATORY DATA ANALYSIS
## Intelligent Cyber Threat Detection & Network Intelligence Platform

**Primary Dataset:** CIC-IDS2017 (Canadian Institute for Cybersecurity)  
**Research Focus:** Network Traffic Feature Analysis, Data Quality Audit, & Machine Learning Readiness

---
### Notebook Objectives
1. Profile network flow characteristics and statistical distributions.
2. Quantify data quality challenges: missing values, division-by-zero infinities, and duplicate records.
3. Analyze extreme class imbalance between benign network flows and diverse attack vectors.
4. Audit multi-collinearity and evaluate network domain feature derivations.
5. Establish ML safeguards against data leakage and target leakage.

*Note on Data Source:* In accordance with platform integrity rules, if the full raw CIC-IDS2017 dataset has not yet been extracted locally into `data/raw/cic_ids2017/`, this notebook seamlessly defaults to the verified synthetic demonstration dataset in `data/sample/`. The active source is transparently identified.*

In [ ]:
import sys
from pathlib import Path

# Configure paths and imports
PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from neuralshield.data.discovery import inspect_dataset_inventory
from neuralshield.data.loader import DatasetLoader
from neuralshield.data.cleaning import (
    normalize_column_names,
    identify_and_drop_redundant_columns,
    sanitize_infinite_and_nan,
    handle_duplicates,
    normalize_labels,
)
from neuralshield.data.validation import audit_dataframe_quality
from neuralshield.analysis.statistics import compute_descriptive_statistics
from neuralshield.analysis.distributions import analyze_class_balance
from neuralshield.analysis.correlations import compute_correlation_matrix, find_high_correlation_pairs
from neuralshield.features.engineering import create_network_domain_features, remove_target_leakage_features

# Set visualization theme
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.sans-serif'] = 'Segoe UI, Inter, sans-serif'
print("Environment and NeuralShield modules loaded successfully.")

### 1. Dataset Inventory & Active File Loading

In [ ]:
raw_dir = PROJECT_ROOT / "data" / "raw" / "cic_ids2017"
sample_file = PROJECT_ROOT / "data" / "sample" / "cic_ids2017_sample.parquet"
sample_csv = PROJECT_ROOT / "data" / "sample" / "cic_ids2017_sample.csv"

inventory = inspect_dataset_inventory(raw_dir)
print(f"Raw benchmark files present: {inventory['expected_present_count']} / {inventory['expected_benchmark_files']}")

loader = DatasetLoader()
if inventory["total_files_found"] > 0:
    first_csv = inventory["expected_inventory"][0]["path"]
    print(f"Loading raw file: {first_csv}")
    df_raw = loader.load_single_file(first_csv, nrows=50000)
    data_mode = "RAW_BENCHMARK"
elif sample_file.is_file():
    print(f"Loading synthetic sample: {sample_file}")
    df_raw = pd.read_parquet(sample_file)
    data_mode = "SYNTHETIC_SAMPLE"
else:
    print(f"Loading CSV sample: {sample_csv}")
    df_raw = pd.read_csv(sample_csv)
    data_mode = "SYNTHETIC_SAMPLE"

print(f"Active Dataset Mode: {data_mode}")
print(f"Dimensions: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

### 2. Dataset Preview & Column Schema

In [ ]:
display(df_raw.head())

In [ ]:
print("Column Data Types Summary:")
print(df_raw.dtypes.value_counts())

### 3. Comprehensive Data Quality Audit
We examine:
- Missing values (NaN).
- Infinite values (+inf, -inf) caused by zero flow durations in `Flow Bytes/s` and `Flow Packets/s`.
- Exact duplicate records that cause data leakage across evaluation splits.
- Redundant duplicate features (`Fwd Header Length.1`).

In [ ]:
audit = audit_dataframe_quality(df_raw, target_col="Label" if "Label" in df_raw.columns else "label")
print(f"Overall Data Quality Score: {audit['quality_score']}/100")
print(f"Total Missing Values: {audit['total_missing_values']}")
print(f"Total Infinite Values: {audit['total_infinite_values']}")
print(f"Exact Duplicate Rows: {audit['duplicate_rows']} ({audit['duplicate_percentage']}%)")
print(f"Memory Footprint: {audit['memory_mb']} MB")

### 4. Data Preprocessing & Sanitization Execution

In [ ]:
# 1. Normalize column names (lowercase, strip whitespace, snake_case)
df_clean, col_mapping = normalize_column_names(df_raw)

# 2. Identify & drop redundant features
df_clean, dropped_cols = identify_and_drop_redundant_columns(df_clean)
print(f"Dropped Redundant Columns: {dropped_cols}")

# 3. Sanitize infinite values
df_clean, inf_stats = sanitize_infinite_and_nan(df_clean, replace_strategy="nan")
print(f"Infinities Sanitized: {inf_stats}")

# 4. Drop exact duplicate rows (leakage prevention)
df_clean, dups_removed = handle_duplicates(df_clean)
print(f"Duplicate Rows Removed: {dups_removed}")

# 5. Normalize target labels
df_clean, label_audit = normalize_labels(df_clean, target_col="label")
print(f"Target Label Normalization: {label_audit['normalized_distribution']}")

# Post-cleaning quality score
clean_audit = audit_dataframe_quality(df_clean, target_col="label")
print(f"Cleaned Dataset Quality Score: {clean_audit['quality_score']}/100")

### 5. Class Imbalance Analysis
Network intrusion datasets inherently exhibit severe class imbalance where benign traffic constitutes >80% of total flows, while certain critical attacks (e.g., Infiltration, Heartbleed) represent rare events (<0.1%).

In [ ]:
balance = analyze_class_balance(df_clean, target_col="label")
print(f"Majority Class: {balance['majority_class']}")
print(f"Minority Class: {balance['minority_class']}")
print(f"Imbalance Ratio: {balance['imbalance_ratio']}:1")

# Visualize Class Balance
plt.figure(figsize=(9, 4))
labels = list(balance["class_counts"].keys())
counts = list(balance["class_counts"].values())
colors = ['#16C784' if 'benign' in str(l).lower() else '#EF4444' for l in labels]

bars = plt.bar(labels, counts, color=colors, edgecolor='#1D2824')
plt.title("Network Flow Class Distribution", fontsize=14, color='#E8F0EB', pad=12)
plt.ylabel("Flow Count", color='#E8F0EB')
plt.xticks(rotation=20, ha='right', color='#8B9A91')
plt.grid(axis='y', color='#1D2824', linestyle='--')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval, f'{yval:,}', ha='center', va='bottom', color='#E8F0EB', fontsize=9)
plt.tight_layout()
plt.show()

### 6. Feature Distributions & Log-Scaling
Network traffic features like `flow_duration` and `flow_bytes_per_sec` exhibit heavy-tailed distributions spanning several orders of magnitude.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

duration_col = next((c for c in df_clean.columns if "duration" in c), None)
if duration_col:
    sns.histplot(df_clean[duration_col], bins=30, ax=axes[0], color='#16C784', kde=True)
    axes[0].set_title(f"Raw Distribution: {duration_col}", color='#E8F0EB')
    axes[0].set_yscale('log')
    axes[0].grid(color='#1D2824')

    log_dur = np.log1p(np.clip(df_clean[duration_col].dropna(), 0, None))
    sns.histplot(log_dur, bins=30, ax=axes[1], color='#38BDF8', kde=True)
    axes[1].set_title(f"Log1p Transformed: {duration_col}", color='#E8F0EB')
    axes[1].grid(color='#1D2824')

plt.tight_layout()
plt.show()

### 7. Correlation Analysis & Collinearity Detection
Detecting redundant features with correlation |r| > 0.90 to avoid multicollinearity in downstream models.

In [ ]:
corr = compute_correlation_matrix(df_clean, max_features=12)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, cbar_kws={'label': 'Pearson Correlation'})
plt.title("Feature Correlation Heatmap", fontsize=14, color='#E8F0EB', pad=12)
plt.xticks(rotation=45, ha='right', color='#8B9A91')
plt.yticks(color='#8B9A91')
plt.tight_layout()
plt.show()

collinear_pairs = find_high_correlation_pairs(corr, threshold=0.90)
print(f"Detected {len(collinear_pairs)} feature pairs with |r| >= 0.90:")
for pair in collinear_pairs[:5]:
    print(f"  - {pair['feature_1']} <--> {pair['feature_2']}: r = {pair['correlation']}")

### 8. Domain Feature Engineering
Deriving packet symmetry ratio, byte symmetry ratio, and SYN density metrics.

In [ ]:
df_engineered = create_network_domain_features(df_clean)
print("New Derived Domain Features:")
new_cols = [c for c in df_engineered.columns if c not in df_clean.columns]
print(new_cols)
display(df_engineered[new_cols].head())

### 9. Target Leakage Safeguards
Ensuring network identifiers (Flow ID, Source IP, Destination IP, Timestamp) are excluded from predictive features.

In [ ]:
df_ml_ready, removed_leaks = remove_target_leakage_features(df_engineered)
print(f"Excluded Leakage Identifiers: {removed_leaks}")
print(f"ML Ready Feature Count: {len(df_ml_ready.columns)}")

## Key Findings & Phase 2 Engineering Recommendations

### Key Insights
1. **Deduplication is Essential:** Real CIC-IDS2017 contains massive exact duplicate traffic. Deduplication prevents test set memorization and artificial accuracy inflation.
2. **Infinite Values Sanitation:** Division-by-zero occurrences in rate metrics (`Flow Bytes/s`, `Flow Packets/s`) must be converted to NaN and imputed or dropped prior to scikit-learn estimation.
3. **Severe Class Imbalance:** Benign flows overwhelm attack categories. Resampling methods (such as SMOTE or class-weighted loss functions) must be strictly restricted to the training folds during cross-validation.
4. **Target Leakage Safeguard:** IP addresses and timestamps must never enter the model feature space.

### Phase 2 Architecture Roadmap
- Establish stratified temporal or group-based train/validation/test splits.
- Train Random Forest and Gradient Boosted Tree baseline classifiers.
- Implement unsupervised Isolation Forest for zero-day anomaly detection.
- Track Precision-Recall AUC (PR-AUC) rather than raw accuracy.